In [ ]:
# Cell 1 - Clone and setup:
repo_url = "https://github.com/siddheshkadane01/Site-Reliability-Server/"
!git clone {repo_url}
repo_dir = repo_url.rstrip("/").split("/")[-1]
%cd $repo_dir

In [ ]:
# Cell 2 - All installs in one cell, correct order:
%pip install -r requirements.txt
%pip install -r requirements_rl.txt

In [ ]:
# Cell 3 - Start FastAPI server:
import subprocess, time, requests
log_file = open("server.log", "w")
server_process = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"],
    stdout=log_file, stderr=log_file
  )
time.sleep(8)
try:
    res = requests.get("http://localhost:7860/health", timeout=5)
    print("✅ Server running" if res.status_code == 200 
          else f"❌ Unexpected status: {res.status_code}")
except Exception as e:
    print("❌ Server failed:", e)
    print(open("server.log").read())

In [ ]:
# Cell 4 - Wandb login using secret:
import wandb, os
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

In [ ]:
# Cell 5 - Train:
import os
os.environ["WANDB_PROJECT"] = "openenv-enterprise-grpo"
os.environ["WANDB_RUN_NAME"] = "grpo-enterprise-42"
os.environ["WANDB_MODE"] = "online"
!python train_grpo.py --epochs 15 --max_steps 25 \
    --num_generations 2 --lora_dropout 0.0 \
    --learning_rate 5e-6 \
    --env_url http://localhost:7860